## Graphs for Netflix Streaming data

The Netflix streaming dataset on Kaggle provides structured information about movies and TV shows available on the platform, including key metadata such as title, type, release year, and content ratings. It typically also includes descriptive attributes like genre, country of production, and a short synopsis, enabling both qualitative and quantitative analysis of Netflix’s catalog. Some versions further enrich the data with external indicators such as IMDb scores, vote counts, and popularity metrics from third-party services. It offers a realistic, moderately large dataset for practicing data cleaning, visualization, and modeling in the context of modern streaming media. [vault.nimc.gov](https://vault.nimc.gov.ng/blog/netflix-dataset-analysis-a-kaggle-exploration-1764797697)

![](img/NetworkX_processing.excalidraw.svg)

Using the terminology of the image above, the Netflix Streaming dataset can be transformed into a "collector/element list".

Before doing that, we carefully clean-up this dataset.

---

### Setup

Imports

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import networkx as nx

Load cleaned dataset.

In [10]:
# Load dataset
df_clean = pd.read_pickle('./datasets/Netflix Streaming Data/Netflix Streaming Data-Cleaned.pkl')
print(df_clean.info())
df_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7955 entries, 0 to 7954
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       7955 non-null   object        
 1   type          7955 non-null   object        
 2   title         7955 non-null   object        
 3   director      5676 non-null   object        
 4   cast          7955 non-null   object        
 5   country       7286 non-null   object        
 6   date_added    7866 non-null   datetime64[ns]
 7   release_year  7955 non-null   int64         
 8   rating        7951 non-null   object        
 9   duration      7952 non-null   object        
 10  listed_in     7955 non-null   object        
 11  description   7955 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 745.9+ KB
None


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
1,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
2,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
3,s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, H...",NaN,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",The arrival of a charismatic young priest brin...
4,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",NaN,2021-09-24,2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...


Consider only entries with non-null "cast" entries

In [11]:
df_cast = df_clean.dropna(subset=['cast'])
df_cast = df_cast.reset_index(drop=True)
df_cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7955 entries, 0 to 7954
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       7955 non-null   object        
 1   type          7955 non-null   object        
 2   title         7955 non-null   object        
 3   director      5676 non-null   object        
 4   cast          7955 non-null   object        
 5   country       7286 non-null   object        
 6   date_added    7866 non-null   datetime64[ns]
 7   release_year  7955 non-null   int64         
 8   rating        7951 non-null   object        
 9   duration      7952 non-null   object        
 10  listed_in     7955 non-null   object        
 11  description   7955 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 745.9+ KB


Consider only entries of type = "Movie".

In [12]:
df_movie = df_cast[df_cast['type'] == 'Movie']
df_movie = df_movie.reset_index(drop=True)
df_movie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5633 entries, 0 to 5632
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       5633 non-null   object        
 1   type          5633 non-null   object        
 2   title         5633 non-null   object        
 3   director      5499 non-null   object        
 4   cast          5633 non-null   object        
 5   country       5265 non-null   object        
 6   date_added    5633 non-null   datetime64[ns]
 7   release_year  5633 non-null   int64         
 8   rating        5631 non-null   object        
 9   duration      5630 non-null   object        
 10  listed_in     5633 non-null   object        
 11  description   5633 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 528.2+ KB


Summary

In [13]:
titles = set(df_movie['title'].unique())
actors = set()
for idx, row in df_movie.iterrows():
    actors.update([actor.strip() for actor in row['cast'].split(',')])
print(f"Total unique titles ('collectors'): {len(titles)}")
print(f"Total unique actors ('elements') : {len(actors)}")

Total unique titles ('collectors'): 5633
Total unique actors ('elements') : 25948


Make list of actors for a sample title.

In [ ]:
actors = df_movie[df_movie['title'] == 'Sankofa']['cast'].values[0].split(',')
actors = [actor.strip() for actor in actors]
actors

---

### Check if dataset can be used to construct bipartite graph

For the moment, we have 5633 data lines. For constructing the bipartite graph, we need unique identifiers for movies.

In [ ]:
print(len(set(df_movie["show_id"])))
print(len(df_movie["show_id"].unique()))

5633
5633


Check whether `title` entries are unique.

In [ ]:
print(len(set(df_movie["title"].str.strip())))
print(len(df_movie["title"].unique()))

5633
5633


In [14]:
movie_titles = set(df_movie["title"].str.strip())
actor_names  = set()

for idx, row in df_movie.iterrows():
    actor_names.update([actor.strip() for actor in row['cast'].split(',')])

print(f"Total number of movie titles: {len(movie_titles)}")
print(f"Total number of actor names: {len(actor_names)}")

Total number of movie titles: 5633
Total number of actor names: 25948


If there is an intersection between `movie_titles` and `actor_names`, we cannot construct a bipartite graph from these two sets.

In [ ]:
intersect = movie_titles.intersection(actor_names)
print(f"Movie titles that are also actor names: {intersect}")

Movie titles that are also actor names: {'Amar', 'Shiva', 'Jimi Hendrix', 'Game', 'Solo', 'Max Rose', 'Secret', 'Tarzan', 'Sebastián Marcelo Wainraich'}


As the intersection between the `movie_titles` set and the `actor_names` set is not null, we have to use the `show_ids` set to construct the bipartite graph. 

---

### Create bipartite graph `B`

In [ ]:
B = nx.Graph()

for idx, row in df_movie.iterrows():
    title = row['title']
    actors = [actor.strip() for actor in row['cast'].split(',')]
    for actor in actors:
        B.add_edge(title, actor)
for node in list(B.nodes(data=True))[200:250]:
    print(node)

#### Check if graph is bipartite

Key concept: A graph is bipartite if and only if it can be colored with 2 colors such that no adjacent nodes have the same color (equivalently, it contains no odd-length cycles).

For the movie-actor graph, B should definitely be bipartite since it connects titles to actors with no actor-to-actor or title-to-title edges.

To check if a graph is bipartite check for odd cycles.

(Bipartite graphs do not contain odd cycles because all edges in such graphs strictly connect distinct, disjoint vertex sets `A` and `B`. Any path starting in `A`, passing through an odd number of edges (e.g., `A` → `B` → `A` → `B`)), must end in a different partition `B`, making it impossible to return to the starting vertex in an odd number of steps.) Example: In the graph below, count the number of edges you pass, if you leave node `1` and return to that node.

![](img/cycles.jpg)

In [ ]:
# If the graph has odd cycles, this will be slow or fail
cycle_basis = nx.cycle_basis(B)
has_odd_cycle = any(len(cycle) % 2 == 1 for cycle in cycle_basis)
if has_odd_cycle:
    print("Graph is NOT bipartite (contains odd cycles)")
else:
    print("Graph is bipartite")

If not: Check for overlap in "title" and "actor" sets.

Overlaps occur if a movie title is also an actor name. 
I silently removed this problem in the original dataset.

In [ ]:
# Nodes that should be titles
titles = set(df_bipartite['title'].unique())

# Nodes that should be actors
actors = set()
for idx, row in df_bipartite.iterrows():
    actors.update([actor.strip() for actor in row['cast'].split(',')])

# Find overlaps
overlap = titles & actors
if overlap:
    print(f"⚠ Found {len(overlap)} nodes in both sets:")
    for node in overlap:
        print(f"  '{node}'")

Next step: Find the titles of those movies that do not share actors with any other movie.

If a movie appears in a connected component with only 1 title node, all of its actors are unique to that movie and don't appear in any other movie in the dataset.

In [ ]:
titles_set = set(df_bipartite['title'].unique())
components_B = list(nx.connected_components(B))

no_of_single_title_components = 0
no_of_actors_in_single_title_components = 0

for component in components_B:
    titles_in_comp = [node for node in component if node in titles_set]
    
    if len(titles_in_comp) == 1:
        no_of_single_title_components += 1
        movie = titles_in_comp[0]
        # Get the cast from the original dataframe
        cast = df_bipartite[df_bipartite['title'] == movie]['cast'].values[0]
        no_of_actors_in_single_title_components += len(cast.split(',')) 
        #print(f"{movie}")
        #print(f"  Cast: {cast}\n")

print(f"Total number of single-title components: {no_of_single_title_components}")
print(f"Total number of actors in single-title components: {no_of_actors_in_single_title_components}\n")

---

### Create projected graph A.

In [ ]:
from networkx.algorithms import bipartite

# Get all actors from the bipartite graph
actors = set()
for idx, row in df_bipartite.iterrows():
    actors.update([actor.strip() for actor in row['cast'].split(',')])

# Project onto actors
A = bipartite.weighted_projected_graph(B, actors)
list(A.edges(data=True))[:10]  # Display first 10 edges with weights

Count connected components in graph A.

In [ ]:
components = list(nx.connected_components(A))
print(f"Number of connected components: {len(components)}\n")

sum_nodes = sum(len(component) for component in components)
print(f"Total number of actor nodes across all connected components: {sum_nodes}")

#for i, component in enumerate(components, 1):
#    print(f"Connected component {i}: {len(component)} nodes")
#    if len(component) <= 5:
#        print(f"  Nodes: {component}")

Find connected components with 5 or less actor nodes.

In [ ]:
titles_set = set(df_bipartite['title'].unique())
components_B = list(nx.connected_components(B))

small_components = [c for c in components_B if len(c) <= 5]
print(f"Found {len(small_components)} components with 5 or less nodes\n")

#for i, component in enumerate(small_components, 1):
#    titles = [n for n in component if n in titles_set]
#    actors = [n for n in component if n not in titles_set]
#    print(f"Component {i} ({len(component)} nodes):")
#    print(f"  Movies: {titles}")
#    print(f"  Actors: {actors}\n")

Create initial visualization and save to file.

In [ ]:
plt.figure(figsize=(200, 200))
pos = nx.spring_layout(A, k=2, iterations=10)
nx.draw(A, pos, with_labels=True, node_color='lightcoral', node_size=300, font_size=6, width=0.5)
plt.savefig('graphs/actor_graph.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

---

### Analyze graph A further.

In [ ]:
degrees = dict(A.degree())
degree_df = pd.DataFrame(list(degrees.items()), columns=['actor', 'degree'])
degree_df = degree_df.sort_values(by='degree', ascending=False)
top_24 = degree_df.head(24)
print (top_24)

# Create and save
plt.figure(figsize=(14, 8))
plt.barh(top_24['actor'], top_24['degree'], color='orange')
plt.xlabel('Degree')
plt.ylabel('Actor')
plt.title('Top 24 Actors by Co-actor Connections')
plt.gca().invert_yaxis()
plt.savefig('graphs/degree_distribution_top24.svg', format='svg', bbox_inches='tight')
plt.show()

![](img/Anupam%20Kher.webp)

Anupam Kher

![](img/Shah_Rukh_Khan.jpg)

Shah Rukh Khan

End of line

In [ ]:
degrees = dict(A.degree())
degree_df = pd.DataFrame(list(degrees.items()), columns=['actor', 'degree'])
degree_df = degree_df.sort_values(by='degree', ascending=False)
bottom_24 = degree_df.tail(24)
print (bottom_24)

# Create and save
plt.figure(figsize=(14, 8))
plt.barh(bottom_24['actor'], bottom_24['degree'], color='orange')
plt.xlabel('Degree')
plt.ylabel('Actor')
plt.title('Bottom 24 Actors by Co-actor Connections')
plt.gca().invert_yaxis()
plt.savefig('graphs/degree_distribution_bottom24.svg', format='svg', bbox_inches='tight')
plt.show()

Who wants to have Ronnie Coleman as co-actor?

![](img/Ronnie_Coleman.png)

---

### Graphviz

![](img/NetworkX_to_svg.png)

We are now generating a graph visualization with Graphviz. This gives us more freedom in graph styling.